# M1 Lab — LLM Foundations for Enterprise Agents

**Track:** Applied Agent Engineering Foundations  
**Module:** M1 — LLM Foundations  
**Environment:** SageMaker Jupyter Notebook  
**Audience:** Software engineers building enterprise agents on AWS

## What learners will learn
By the end of this lab, learners will be able to:
1. Connect to Bedrock from SageMaker.
2. Compare prompt styles and understand why prompt structure matters.
3. Turn a user request into a structured action plan.
4. Simulate prompt-to-action pipelines with lightweight backend functions.
5. Add basic validation and error handling before production use.

## Instructor flow
- Part A: Environment and model access
- Part B: Prompting patterns
- Part C: Structured outputs
- Part D: Prompt-to-action pipeline
- Part E: Mini-hack

> **Explain to learners:** In enterprise agents, the model is only one part of the system. Reliability comes from how we structure prompts, validate outputs, route actions, and handle failures.


## Before you run
Update the config cell below if you want different model IDs.

This notebook assumes:
- SageMaker notebook has AWS credentials attached through an execution role
- Bedrock model access is already enabled
- You want a lightweight, classroom-friendly example that can be extended later


In [ ]:

# Uncomment only if your environment is missing any package
# %pip install -q boto3 pandas

import json
import boto3
from botocore.exceptions import ClientError

BEDROCK_TEXT_MODEL = "amazon.nova-lite-v1:0"
AWS_REGION = boto3.Session().region_name or "us-east-1"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

print("Region:", AWS_REGION)
print("Configured text model:", BEDROCK_TEXT_MODEL)


## Step 1 — Quick connectivity check

**Goal:** Confirm that SageMaker can talk to Bedrock.

**Discuss:**
- Why SageMaker execution roles matter
- Why enterprise labs should start with a simple smoke test
- Why we keep the first call small and cheap


In [ ]:

def ask_llm(user_text: str,
            system_text: str = "You are a helpful enterprise AI assistant.",
            max_tokens: int = 250,
            temperature: float = 0.2) -> str:
    response = bedrock_runtime.converse(
        modelId=BEDROCK_TEXT_MODEL,
        system=[{"text": system_text}],
        messages=[
            {
                "role": "user",
                "content": [{"text": user_text}]
            }
        ],
        inferenceConfig={
            "maxTokens": max_tokens,
            "temperature": temperature
        }
    )
    return response["output"]["message"]["content"][0]["text"]

print(ask_llm("In two lines, explain what an AI agent is."))


## Step 2 — Compare prompt styles

We will test three common patterns:
1. Simple instruction
2. Role-based instruction
3. Structured output request

**Explain to learners:** Better prompts do not mean longer prompts. They mean clearer intent, better constraints, and better output shape.


In [ ]:

prompts = {
    "simple_instruction": "Summarize why error handling matters in enterprise AI applications.",
    "role_based": (
        "You are a senior platform architect. "
        "Summarize why error handling matters in enterprise AI applications "
        "for a team of backend engineers."
    ),
    "structured": (
        "Explain why error handling matters in enterprise AI applications. "
        "Return the answer as JSON with keys: risk, impact, mitigation."
    )
}

for name, prompt in prompts.items():
    print("=" * 80)
    print("PROMPT STYLE:", name)
    print(ask_llm(prompt))
    print()


## Reflection checkpoint

Discuss these questions before moving on:
- Which output would you trust more in an application pipeline?
- Which output is easiest to parse downstream?
- Which output would be easiest to test automatically?


## Step 3 — Ask the model for an action plan

A common agent pattern is:
1. Understand the request
2. Convert it into a structured plan
3. Validate the plan
4. Execute the allowed actions

We will first ask the model to return a JSON plan.


In [ ]:

ACTION_PLANNER_PROMPT = '''
You are an action planner for an enterprise assistant.

Return valid JSON only.
Use this schema:
{
  "intent": "<one short label>",
  "needs_tool": true or false,
  "tool_name": "<tool or none>",
  "arguments": { ... },
  "reason_for_tool": "<one sentence>"
}

Allowed tool names:
- create_ticket
- get_policy
- none

Examples:
User: "Create an HR ticket for VPN access issue"
Output:
{"intent":"create_hr_ticket","needs_tool":true,"tool_name":"create_ticket","arguments":{"category":"IT Support","summary":"VPN access issue"},"reason_for_tool":"A ticket must be created in the backend system."}

User: "What is the leave policy?"
Output:
{"intent":"policy_question","needs_tool":true,"tool_name":"get_policy","arguments":{"topic":"leave policy"},"reason_for_tool":"The answer should come from enterprise policy content."}
'''

def plan_action(user_request: str) -> dict:
    raw = ask_llm(
        user_text=f"{ACTION_PLANNER_PROMPT}\n\nUser request: {user_request}",
        system_text="You convert user requests into safe structured action plans.",
        max_tokens=300,
        temperature=0
    )
    return json.loads(raw)

plan_action("Create an HR ticket for laptop replacement")


## Step 4 — Validate model output before execution

**Explain to learners:** Never trust raw model output blindly. Even with structured prompting, validate:
- required keys
- allowed tool names
- argument types
- missing values


In [ ]:

ALLOWED_TOOLS = {"create_ticket", "get_policy", "none"}

def validate_plan(plan: dict) -> None:
    required_keys = {"intent", "needs_tool", "tool_name", "arguments", "reason_for_tool"}
    missing = required_keys - set(plan.keys())
    if missing:
        raise ValueError(f"Missing keys: {missing}")

    if plan["tool_name"] not in ALLOWED_TOOLS:
        raise ValueError(f"Disallowed tool: {plan['tool_name']}")

    if not isinstance(plan["arguments"], dict):
        raise ValueError("'arguments' must be a dictionary")

test_plan = plan_action("What is the leave policy?")
validate_plan(test_plan)
test_plan


## Step 5 — Simulate backend tools

For the classroom, we keep the tools simple and visible.
Later, the same pattern can call APIs, databases, ticketing systems, or internal services.


In [ ]:

# In-memory demo data
ticket_store = []
policy_store = {
    "leave policy": "Employees can apply planned leave through the HR portal with manager approval.",
    "remote work": "Remote work is allowed for approved roles and depends on team policy.",
    "contractor policy": "Contractors must follow the contract-specific access and compliance rules."
}

def create_ticket(category: str, summary: str) -> str:
    ticket_id = f"TKT-{len(ticket_store)+1:03d}"
    ticket_store.append({
        "ticket_id": ticket_id,
        "category": category,
        "summary": summary
    })
    return f"Ticket created successfully: {ticket_id} | Category: {category} | Summary: {summary}"

def get_policy(topic: str) -> str:
    return policy_store.get(topic.lower(), "Policy not found in the demo policy store.")


## Step 6 — Build a prompt-to-action pipeline

This is the core classroom takeaway:
- LLM decides the intent
- Code validates the plan
- Code calls the allowed function
- Final answer is assembled safely


In [ ]:

def execute_plan(plan: dict) -> str:
    validate_plan(plan)

    tool_name = plan["tool_name"]
    args = plan["arguments"]

    if tool_name == "create_ticket":
        return create_ticket(
            category=args.get("category", "General"),
            summary=args.get("summary", "No summary provided")
        )
    elif tool_name == "get_policy":
        return get_policy(topic=args.get("topic", ""))
    return "No tool execution required."

def run_foundation_agent(user_request: str) -> dict:
    plan = plan_action(user_request)
    execution_result = execute_plan(plan)
    return {
        "user_request": user_request,
        "plan": plan,
        "execution_result": execution_result
    }

demo_result = run_foundation_agent("Create an HR ticket for VPN access issue")
demo_result


## Step 7 — Add lightweight error handling

Enterprise agents should fail safely and clearly.


In [ ]:

def safe_run_foundation_agent(user_request: str) -> dict:
    try:
        result = run_foundation_agent(user_request)
        result["status"] = "ok"
        return result
    except (ValueError, KeyError, json.JSONDecodeError) as e:
        return {
            "status": "validation_error",
            "user_request": user_request,
            "error": str(e)
        }
    except ClientError as e:
        return {
            "status": "aws_error",
            "user_request": user_request,
            "error": str(e)
        }
    except Exception as e:
        return {
            "status": "unexpected_error",
            "user_request": user_request,
            "error": str(e)
        }

safe_run_foundation_agent("What is the leave policy?")


## Mini-hack — Build a structured conversational agent

### Task
Extend the current pipeline to support one more business workflow.

### Suggested options
- Reset password request
- Access request
- Leave balance inquiry
- Travel approval request

### Minimum success criteria
1. Add one new tool.
2. Update the planner prompt so the tool can be selected.
3. Validate the new tool name.
4. Show one successful run and one failure-safe run.

### Stretch goal
Return a final user response that contains:
- action taken
- tool used
- next step for the employee


In [ ]:

# --- STUDENT WORK AREA ---
# Add your new tool here and update the planner / executor logic above.


## Wrap-up

Learners should leave this notebook understanding that:
- prompts are part of software design, not just natural language
- structured outputs are easier to validate than free text
- tool routing should always be validated before execution
- safe failure behavior is part of production readiness
